# Load libraries

In [1]:
suppressMessages(library("data.table"))
suppressMessages(library("withr"))
suppressMessages(library("ggplot2"))
suppressMessages(library("farver"))
suppressMessages(library("labeling"))
suppressMessages(library("optparse"))
suppressMessages(library("dplyr"))
suppressMessages(library("withr"))
suppressMessages(library("backports"))
suppressMessages(library("broom"))
suppressMessages(library("rstudioapi"))
suppressMessages(library("tzdb"))
suppressMessages(library("svglite"))
suppressMessages(library("ggeasy"))
suppressMessages(library("tidyverse"))
suppressMessages(library("BiocGenerics"))
suppressMessages(library("S4Vectors"))
suppressMessages(library("IRanges"))
suppressMessages(library("GenomeInfoDb"))
suppressMessages(library("GenomicRanges"))
suppressMessages(library("Biobase"))
suppressMessages(library("AnnotationDbi"))
suppressMessages(library("GO.db"))
suppressMessages(library("org.Hs.eg.db"))
suppressMessages(library("ggrepel"))
suppressMessages(library("RColorBrewer"))
suppressMessages(library("svglite"))
suppressMessages(library("cowplot"))
suppressMessages(library("Matrix"))
suppressMessages(library("rtracklayer"))
suppressMessages(library("Biostrings"))
suppressMessages(library("ggrepel"))
suppressMessages(library("ggnewscale"))
suppressMessages(library("splitstackshape"))
suppressMessages(library("viridis"))
suppressMessages(library("viridis"))
suppressMessages(library("ggsci"))
suppressMessages(library("plyr"))
suppressMessages(library("ggupset"))
suppressMessages(library("patchwork"))
suppressMessages(library("ggh4x"))
suppressMessages(library("ComplexHeatmap"))
suppressMessages(library("tidyr"))
suppressMessages(library("circlize"))
suppressMessages(library("grid"))
suppressMessages(library("ggrepel"))
suppressMessages(library("cluster"))
suppressMessages(library("clusterProfiler"))
suppressMessages(library("projectStyleR"))


## Load palettes

In [2]:
palettes_path <- system.file("palettes.yaml", package = "projectStyleR")
themes_path <- system.file("themes.yaml", package = "projectStyleR")

In [3]:
palettes <- yaml::read_yaml(palettes_path)
themes <- yaml::read_yaml(themes_path)

In [4]:
available_palettes()
available_themes()

Available palettes:

 - primary

 - vibrant

 - muted_tones

Available themes:

 - default

 - publication



# 0. Read in eQTLs

In [5]:
# optional, only if you want to check later
eqtl_full <- fread("/nfs/team151/mt19/Whole_cis_trans_graph_ta_label_collapsed.tsv")

In [6]:
GWAS_colocalized_eqtl_non_collapsed <- fread("/nfs/users/nfs_m/mt19/cardinal_analysis/ht/datasets/Flanders/trans_eQTLS_ERs/All_complex_trait_cis_trans_relationships.tsv", sep="\t", header=TRUE)

In [7]:
cat("GWAS_colocalized_eqtl_non_collapsed\n")
cat(str(GWAS_colocalized_eqtl_non_collapsed))
cat("\n")


GWAS_colocalized_eqtl_non_collapsed
Classes ‘data.table’ and 'data.frame':	19563 obs. of  11 variables:
 $ Source_trait : chr  "T_CD8_EM:ENSG00000165689" "T_CD8_EM:ENSG00000165689" "T_CD8_EM:ENSG00000165689" "T_CD8_EM:ENSG00000165689" ...
 $ Target_trait : chr  "T_CD4_CM:ENSG00000198931" "T_CD4_CM:ENSG00000130429" "T_CD4_Treg:ENSG00000130429" "T_CD8_CM:ENSG00000130429" ...
 $ Source_cell  : chr  "T_CD8_EM" "T_CD8_EM" "T_CD8_EM" "T_CD8_EM" ...
 $ Source_gene  : chr  "ENSG00000165689" "ENSG00000165689" "ENSG00000165689" "ENSG00000165689" ...
 $ Target_cell  : chr  "T_CD4_CM" "T_CD4_CM" "T_CD4_Treg" "T_CD8_CM" ...
 $ Target_gene  : chr  "ENSG00000198931" "ENSG00000130429" "ENSG00000130429" "ENSG00000130429" ...
 $ Source_module: chr  "M_21217" "M_21217" "M_21217" "M_21217" ...
 $ edge.type    : chr  "Across" "Across" "Across" "Across" ...
 $ ta.label     : chr  "mean corpuscular hemoglobin concentration" "mean corpuscular hemoglobin concentration" "mean corpuscular hemoglobin concentratio

In [8]:
colnames(GWAS_colocalized_eqtl_non_collapsed)

[1] "Source_trait"  "Target_trait"  "Source_cell"   "Source_gene"  
 [5] "Target_cell"   "Target_gene"   "Source_module" "edge.type"    
 [9] "ta.label"      "edge.code"     "disease.type"

In [9]:
whole_eqtl_full <- fread("/nfs/team151/mt19/Whole_cis_tras_graph.tsv")

In [10]:
cat("whole_eqtl_full\n")
str(whole_eqtl_full)
cat("\n")


whole_eqtl_full
Classes ‘data.table’ and 'data.frame':	266992 obs. of  8 variables:
 $ Source_trait : chr  "Monocyte_CD14:ENSG00000230074" "Monocyte_CD14:ENSG00000230074" "Monocyte_CD14:ENSG00000230074" "Monocyte_CD14:ENSG00000230074" ...
 $ Target_trait : chr  "Monocyte_CD14:ENSG00000119922" "Monocyte_CD14:ENSG00000119917" "Monocyte_CD14:ENSG00000185745" "Monocyte_CD14:ENSG00000185885" ...
 $ Source_cell  : chr  "Monocyte_CD14" "Monocyte_CD14" "Monocyte_CD14" "Monocyte_CD14" ...
 $ Source_gene  : chr  "ENSG00000230074" "ENSG00000230074" "ENSG00000230074" "ENSG00000230074" ...
 $ Target_cell  : chr  "Monocyte_CD14" "Monocyte_CD14" "Monocyte_CD14" "Monocyte_CD14" ...
 $ Target_gene  : chr  "ENSG00000119922" "ENSG00000119917" "ENSG00000185745" "ENSG00000185885" ...
 $ Source_module: chr  "M_19657" "M_19657" "M_19657" "M_19657" ...
 $ edge.type    : chr  "Within" "Within" "Within" "Within" ...
 - attr(*, ".internal.selfref")=<externalptr> 



In [11]:
colnames(whole_eqtl_full)

[1] "Source_trait"  "Target_trait"  "Source_cell"   "Source_gene"  
[5] "Target_cell"   "Target_gene"   "Source_module" "edge.type"

In [12]:
whole_eqtl_full_with_program_ID<- fread("/nfs/team151/mt19/Whole_cis_tras_graph_with_programs.tsv")

In [13]:
cat("whole_eqtl_full_with_program_ID\n")
str(whole_eqtl_full_with_program_ID)
cat("\n")

whole_eqtl_full_with_program_ID
Classes ‘data.table’ and 'data.frame':	305496 obs. of  9 variables:
 $ Source_trait : chr  "Monocyte_CD14:ENSG00000230074" "Monocyte_CD14:ENSG00000230074" "Monocyte_CD14:ENSG00000230074" "Monocyte_CD14:ENSG00000230074" ...
 $ Target_trait : chr  "Monocyte_CD14:ENSG00000119922" "Monocyte_CD14:ENSG00000119922" "Monocyte_CD14:ENSG00000119917" "Monocyte_CD14:ENSG00000119917" ...
 $ program      : chr  "D_Factor286" "D_Factor286" "D_Factor286" "D_Factor286" ...
 $ Source_cell  : chr  "Monocyte_CD14" "Monocyte_CD14" "Monocyte_CD14" "Monocyte_CD14" ...
 $ Source_gene  : chr  "ENSG00000230074" "ENSG00000230074" "ENSG00000230074" "ENSG00000230074" ...
 $ Target_cell  : chr  "Monocyte_CD14" "Monocyte_CD14" "Monocyte_CD14" "Monocyte_CD14" ...
 $ Target_gene  : chr  "ENSG00000119922" "ENSG00000119922" "ENSG00000119917" "ENSG00000119917" ...
 $ Source_module: chr  "M_19657" "M_19657" "M_19657" "M_19657" ...
 $ edge.type    : chr  "Within" "Within" "Within" "Within" .

In [14]:
colnames(whole_eqtl_full_with_program_ID)

[1] "Source_trait"  "Target_trait"  "program"       "Source_cell"  
[5] "Source_gene"   "Target_cell"   "Target_gene"   "Source_module"
[9] "edge.type"

# PATH

In [15]:
outdir <- "/nfs/team151/mt19/overhaul_classification_factors_with_programs/"
dir.create(outdir, recursive = TRUE, showWarnings = FALSE)



cat("Created:", outdir, "\n")

Created: /nfs/team151/mt19/overhaul_classification_factors_with_programs/ 


# Classification 1:Factor cell-type topology

## checks

In [16]:

# 1. Does edge.type == "Within" always mean Source_cell == Target_cell,
#    and edge.type == "Across" always mean Source_cell != Target_cell?

check_tab <- whole_eqtl_full_with_program_ID[, .N, by = .(edge.type, same_cell = Source_cell == Target_cell)]
print(check_tab)

# 2. Flag any violations explicitly (should be 0 rows if the column is defined as expected)
violations <- whole_eqtl_full_with_program_ID[
  (edge.type == "Within" & Source_cell != Target_cell) |
  (edge.type == "Across" & Source_cell == Target_cell)
]
cat("Number of violating rows:", nrow(violations), "\n")
if (nrow(violations) > 0) print(head(violations))

   edge.type same_cell      N
      <char>    <lgcl>  <int>
1:    Within      TRUE  93785
2:    Across     FALSE 211711
Number of violating rows: 0 


## Classification heuristic

In [17]:
# --- Reload fresh (or strip any prior merge artifacts before re-running) ---
if ("edge_type_composition" %in% names(whole_eqtl_full_with_program_ID)) {
  whole_eqtl_full_with_program_ID[, edge_type_composition := NULL]
}
if ("edge_type_composition.x" %in% names(whole_eqtl_full_with_program_ID)) {
  whole_eqtl_full_with_program_ID[, c("edge_type_composition.x", "edge_type_composition.y") := NULL]
}

# If in doubt, safest is a full fresh reload instead:
# whole_eqtl_full_with_program_ID <- fread("/nfs/team151/mt19/Whole_cis_tras_graph_with_programs.tsv")

whole_eqtl_full_with_program_ID[, same_cell := Source_cell == Target_cell]
stopifnot(nrow(whole_eqtl_full_with_program_ID[(edge.type == "Within" & !same_cell) |
                                (edge.type == "Across" &  same_cell)]) == 0)

factor_topology <- whole_eqtl_full_with_program_ID[
  , .(
      n_edges        = .N,
      n_within       = sum(edge.type == "Within"),
      n_across       = sum(edge.type == "Across"),
      n_source_cells = uniqueN(Source_cell)
    ),
  by = .(Source_module, Source_gene, program)
]

factor_topology[, edge_type_composition := fcase(
  n_within > 0 & n_across == 0, "Within-only",
  n_across > 0 & n_within == 0, "Across-only",
  n_within > 0 & n_across > 0,  "Mixed"
)]

print(factor_topology[, .N, by = edge_type_composition])

n_rows_before_merge <- nrow(whole_eqtl_full_with_program_ID)

whole_eqtl_full_with_program_ID <- merge(
  whole_eqtl_full_with_program_ID,
  factor_topology[, .(Source_module, Source_gene, program, edge_type_composition)],
  by = c("Source_module", "Source_gene", "program"),
  all.x = TRUE
)

# --- Explicit check: merge must be 1-to-1, row count must not change ---
cat("Rows before merge:", n_rows_before_merge, "| after merge:", nrow(whole_eqtl_full_with_program_ID), "\n")
stopifnot(nrow(whole_eqtl_full_with_program_ID) == n_rows_before_merge)
stopifnot(!any(grepl("\\.x$|\\.y$", names(whole_eqtl_full_with_program_ID))))
cat("Clean merge confirmed: single edge_type_composition column, no row duplication.\n")

   edge_type_composition     N
                  <char> <int>
1:           Within-only  2018
2:           Across-only  1149
Rows before merge: 305496 | after merge: 305496 
Clean merge confirmed: single edge_type_composition column, no row duplication.


In [18]:
str(whole_eqtl_full_with_program_ID)

Classes ‘data.table’ and 'data.frame':	305496 obs. of  11 variables:
 $ Source_module        : chr  "M_10" "M_10" "M_10" "M_10" ...
 $ Source_gene          : chr  "ENSG00000103145" "ENSG00000103145" "ENSG00000103145" "ENSG00000103145" ...
 $ program              : chr  "D_Factor36" "D_Factor36" "D_Factor36" "D_Factor36" ...
 $ Source_trait         : chr  "T_CD4_naive:ENSG00000103145" "T_CD4_naive:ENSG00000103145" "T_CD4_naive:ENSG00000103145" "T_CD4_naive:ENSG00000103145" ...
 $ Target_trait         : chr  "T_CD4_naive:ENSG00000185885" "T_CD4_naive:ENSG00000117984" "T_CD4_naive:ENSG00000175390" "T_CD4_naive:ENSG00000090554" ...
 $ Source_cell          : chr  "T_CD4_naive" "T_CD4_naive" "T_CD4_naive" "T_CD4_naive" ...
 $ Target_cell          : chr  "T_CD4_naive" "T_CD4_naive" "T_CD4_naive" "T_CD4_naive" ...
 $ Target_gene          : chr  "ENSG00000185885" "ENSG00000117984" "ENSG00000175390" "ENSG00000090554" ...
 $ edge.type            : chr  "Within" "Within" "Within" "Within" ...
 $ s

# Classification 2: Edge-level classification

In [19]:
library(data.table)

# --- Reload base eQTL table and all annotation resources fresh ---
# whole_eqtl_full_with_program_ID <- readRDS("path/to/whole_eqtl_full_with_program_ID.rds")  # adjust as needed

collectri_strict   <- fread("/nfs/team151/mt19/annotation_preprocessed/collectri_strict.tsv")
postar3_strict     <- fread("/nfs/team151/mt19/annotation_preprocessed/postar3_strict.tsv")
liana_lenient      <- fread("/nfs/team151/mt19/annotation_preprocessed/liana_lenient.tsv")
string_exp_strict  <- fread("/nfs/team151/mt19/annotation_preprocessed/string_experimental_strict.tsv")
string_exp_lenient <- fread("/nfs/team151/mt19/annotation_preprocessed/string_experimental_lenient.tsv")
merops_strict      <- fread("/nfs/team151/mt19/annotation_preprocessed/merops_strict.tsv")

# --- Sanity checks (unchanged from before) ---
stopifnot(all(collectri_strict$partner1_role == "TF"), all(collectri_strict$partner2_role == "TF_target"))
stopifnot(all(postar3_strict$partner1_role == "RBP"), all(postar3_strict$partner2_role == "RBP_target"))
stopifnot(all(liana_lenient$partner1_role == "Ligand"), all(liana_lenient$partner2_role == "Receptor"))
stopifnot(all(string_exp_strict$partner1_role == "Arbitrary"), all(string_exp_strict$directed == FALSE))
stopifnot(all(string_exp_lenient$partner1_role == "Arbitrary"), all(string_exp_lenient$directed == FALSE))
stopifnot(all(merops_strict$partner1_role == "Peptidase"), all(merops_strict$partner2_role == "Substrate_peptidase"))

# --- Helper: append a label + its resource tag in lockstep, for a given row mask ---
append_role <- function(edge_role_col, edge_resource_col, mask, label, resource_tag) {
  edge_role_col[mask]     <- lapply(edge_role_col[mask],     function(x) c(x, label))
  edge_resource_col[mask] <- lapply(edge_resource_col[mask], function(x) c(x, resource_tag))
  list(role = edge_role_col, resource = edge_resource_col)
}

# --- Build edge_roles table with two parallel list-columns ---
edge_roles <- copy(whole_eqtl_full_with_program_ID)
edge_roles[, edge_role     := vector("list", .N)]
edge_roles[, edge_resource := vector("list", .N)]

edge_roles[, pair_key_fwd := paste(Source_gene, Target_gene)]
edge_roles[, pair_key_rev := paste(Target_gene, Source_gene)]
edge_roles[, pair_key_unordered := paste(pmin(Source_gene, Target_gene), pmax(Source_gene, Target_gene))]

# --- Pair-key sets, each carrying its own resource tag pulled from the actual loaded table ---
tf_target_pairs   <- collectri_strict[, paste(partner1_ensembl_gene_id, partner2_ensembl_gene_id)]
tf_tag             <- paste0("CollecTRI (", collectri_strict$stringency_detail[1], ")")

rbp_target_pairs  <- postar3_strict[, paste(partner1_ensembl_gene_id, partner2_ensembl_gene_id)]
rbp_tag            <- paste0("POSTAR3 (", postar3_strict$stringency_detail[1], ")")

lr_pairs          <- liana_lenient[, paste(partner1_ensembl_gene_id, partner2_ensembl_gene_id)]
lr_tag             <- paste0("Liana (", liana_lenient$stringency_detail[1], ")")

peptidase_pairs   <- merops_strict[, paste(partner1_ensembl_gene_id, partner2_ensembl_gene_id)]
peptidase_tag      <- paste0("MEROPS (", merops_strict$stringency_detail[1], ")")

string_strict_pairs <- string_exp_strict[, paste(pmin(partner1_ensembl_gene_id, partner2_ensembl_gene_id),
                                                   pmax(partner1_ensembl_gene_id, partner2_ensembl_gene_id))]
string_strict_tag  <- paste0("STRING_experimental (", string_exp_strict$stringency_detail[1], ")")

string_exp_lenient[, pair_key_unordered := paste(pmin(partner1_ensembl_gene_id, partner2_ensembl_gene_id),
                                                   pmax(partner1_ensembl_gene_id, partner2_ensembl_gene_id))]
string_lenient_only_pairs <- unique(string_exp_lenient[!pair_key_unordered %in% string_strict_pairs]$pair_key_unordered)
string_lenient_tag <- "STRING_experimental (score_experimentally_determined_interaction >=0.1 <0.4)"

# --- Apply all ten label/resource appends, in lockstep ---
r <- append_role(edge_roles$edge_role, edge_roles$edge_resource, edge_roles$pair_key_fwd %in% tf_target_pairs, "Source_TF->Target", tf_tag)
edge_roles[, `:=`(edge_role = r$role, edge_resource = r$resource)]

r <- append_role(edge_roles$edge_role, edge_roles$edge_resource, edge_roles$pair_key_rev %in% tf_target_pairs, "Target_TF->Source", tf_tag)
edge_roles[, `:=`(edge_role = r$role, edge_resource = r$resource)]

r <- append_role(edge_roles$edge_role, edge_roles$edge_resource, edge_roles$pair_key_fwd %in% rbp_target_pairs, "Source_RBP->Target", rbp_tag)
edge_roles[, `:=`(edge_role = r$role, edge_resource = r$resource)]

r <- append_role(edge_roles$edge_role, edge_roles$edge_resource, edge_roles$pair_key_rev %in% rbp_target_pairs, "Target_RBP->Source", rbp_tag)
edge_roles[, `:=`(edge_role = r$role, edge_resource = r$resource)]

r <- append_role(edge_roles$edge_role, edge_roles$edge_resource, edge_roles$pair_key_fwd %in% lr_pairs, "Source_Ligand->Target_Receptor", lr_tag)
edge_roles[, `:=`(edge_role = r$role, edge_resource = r$resource)]

r <- append_role(edge_roles$edge_role, edge_roles$edge_resource, edge_roles$pair_key_rev %in% lr_pairs, "Target_Ligand->Source_Receptor", lr_tag)
edge_roles[, `:=`(edge_role = r$role, edge_resource = r$resource)]

r <- append_role(edge_roles$edge_role, edge_roles$edge_resource, edge_roles$pair_key_fwd %in% peptidase_pairs, "Source_Peptidase->Target_Substrate", peptidase_tag)
edge_roles[, `:=`(edge_role = r$role, edge_resource = r$resource)]

r <- append_role(edge_roles$edge_role, edge_roles$edge_resource, edge_roles$pair_key_rev %in% peptidase_pairs, "Target_Peptidase->Source_Substrate", peptidase_tag)
edge_roles[, `:=`(edge_role = r$role, edge_resource = r$resource)]

r <- append_role(edge_roles$edge_role, edge_roles$edge_resource, edge_roles$pair_key_unordered %in% string_strict_pairs, "PPI (STRING_experimental >=0.4)", string_strict_tag)
edge_roles[, `:=`(edge_role = r$role, edge_resource = r$resource)]

r <- append_role(edge_roles$edge_role, edge_roles$edge_resource, edge_roles$pair_key_unordered %in% string_lenient_only_pairs, "PPI (STRING_experimental >=0.1 <0.4)", string_lenient_tag)
edge_roles[, `:=`(edge_role = r$role, edge_resource = r$resource)]

edge_roles[, n_edge_roles := lengths(edge_role)]

# --- Validation: cross-check against the label frequency table from the last confirmed run ---
print(sort(table(unlist(edge_roles$edge_role)), decreasing = TRUE))
print(edge_roles[, .N, by = n_edge_roles])

# --- Positional-correspondence spot check: role and resource lists must always be same length ---
mismatch <- sum(lengths(edge_roles$edge_role) != lengths(edge_roles$edge_resource))
cat("Rows where edge_role and edge_resource lengths differ (should be 0):", mismatch, "\n")


PPI (STRING_experimental >=0.1 <0.4)                    Source_TF->Target 
                                1857                                  422 
                  Target_RBP->Source      PPI (STRING_experimental >=0.4) 
                                 286                                  219 
      Target_Ligand->Source_Receptor                   Source_RBP->Target 
                                 140                                   90 
                   Target_TF->Source       Source_Ligand->Target_Receptor 
                                  89                                   14 
  Target_Peptidase->Source_Substrate 
                                  11 
   n_edge_roles      N
          <int>  <int>
1:            0 302416
2:            1   3034
3:            2     44
4:            3      2
Rows where edge_role and edge_resource lengths differ (should be 0): 0 


In [20]:
# --- Option 1: temporary display-only columns (collapsed strings), doesn't touch the real list-columns ---
edge_roles[n_edge_roles > 1, .(
  edge_role_display     = sapply(edge_role, paste, collapse = " | "),
  edge_resource_display = sapply(edge_resource, paste, collapse = " | ")
)]

# --- Option 2: row-by-row cat, fully unambiguous, good for a handful of rows ---
multi_label_rows <- edge_roles[n_edge_roles > 1]
for (i in seq_len(nrow(multi_label_rows))) {
  cat(sprintf("Row %d: Source_gene=%s, Target_gene=%s\n",
              i, multi_label_rows$Source_gene[i], multi_label_rows$Target_gene[i]))
  cat("  edge_role:    ", paste(multi_label_rows$edge_role[[i]], collapse = " | "), "\n")
  cat("  edge_resource:", paste(multi_label_rows$edge_resource[[i]], collapse = " | "), "\n\n")
}

edge_role_display,edge_resource_display
<chr>,<chr>
Target_Ligand->Source_Receptor | PPI (STRING_experimental >=0.4),Liana (Liana consensus resource) | STRING_experimental (score_experimentally_determined_interaction >= 0.4)
Target_Ligand->Source_Receptor | PPI (STRING_experimental >=0.4),Liana (Liana consensus resource) | STRING_experimental (score_experimentally_determined_interaction >= 0.4)
Target_Ligand->Source_Receptor | PPI (STRING_experimental >=0.4),Liana (Liana consensus resource) | STRING_experimental (score_experimentally_determined_interaction >= 0.4)
Target_Ligand->Source_Receptor | PPI (STRING_experimental >=0.4),Liana (Liana consensus resource) | STRING_experimental (score_experimentally_determined_interaction >= 0.4)
Source_Ligand->Target_Receptor | PPI (STRING_experimental >=0.1 <0.4),Liana (Liana consensus resource) | STRING_experimental (score_experimentally_determined_interaction >=0.1 <0.4)
Source_Ligand->Target_Receptor | PPI (STRING_experimental >=0.1 <0.4),Liana (Liana consensus resource) | STRING_experimental (score_experimentally_determined_interaction >=0.1 <0.4)
Source_Ligand->Target_Receptor | PPI (STRING_experimental >=0.1 <0.4),Liana (Liana consensus resource) | STRING_experimental (score_experimentally_determined_interaction >=0.1 <0.4)
Source_TF->Target | Target_TF->Source | PPI (STRING_experimental >=0.1 <0.4),CollecTRI (>=2 CollecTRI sub-resources) | CollecTRI (>=2 CollecTRI sub-resources) | STRING_experimental (score_experimentally_determined_interaction >=0.1 <0.4)
Source_TF->Target | Target_TF->Source | PPI (STRING_experimental >=0.1 <0.4),CollecTRI (>=2 CollecTRI sub-resources) | CollecTRI (>=2 CollecTRI sub-resources) | STRING_experimental (score_experimentally_determined_interaction >=0.1 <0.4)


Row 1: Source_gene=ENSG00000160791, Target_gene=ENSG00000271503
  edge_role:     Target_Ligand->Source_Receptor | PPI (STRING_experimental >=0.4) 
  edge_resource: Liana (Liana consensus resource) | STRING_experimental (score_experimentally_determined_interaction >= 0.4) 

Row 2: Source_gene=ENSG00000160791, Target_gene=ENSG00000271503
  edge_role:     Target_Ligand->Source_Receptor | PPI (STRING_experimental >=0.4) 
  edge_resource: Liana (Liana consensus resource) | STRING_experimental (score_experimentally_determined_interaction >= 0.4) 

Row 3: Source_gene=ENSG00000160791, Target_gene=ENSG00000271503
  edge_role:     Target_Ligand->Source_Receptor | PPI (STRING_experimental >=0.4) 
  edge_resource: Liana (Liana consensus resource) | STRING_experimental (score_experimentally_determined_interaction >= 0.4) 

Row 4: Source_gene=ENSG00000160791, Target_gene=ENSG00000271503
  edge_role:     Target_Ligand->Source_Receptor | PPI (STRING_experimental >=0.4) 
  edge_resource: Liana (Liana c

## sparsity check

In [21]:
# --- Edge-level sparsity, overall and per resource/direction ---
n_edges_total <- nrow(edge_roles)

cat("=== Overall edge-level sparsity ===\n")
overall <- edge_roles[, .N, by = .(has_annotation = n_edge_roles > 0)]
overall[, pct := round(100 * N / n_edges_total, 4)]
print(overall)

cat("\n=== Per-label edge counts and % of total edges ===\n")
label_counts <- sort(table(unlist(edge_roles$edge_role)), decreasing = TRUE)
label_pct <- data.table(
  label   = names(label_counts),
  n_edges = as.integer(label_counts),
  pct_of_all_edges = round(100 * as.integer(label_counts) / n_edges_total, 4)
)
print(label_pct)

# --- Per-resource sparsity (collapsing directions, since resource is what matters for the permutation test) ---
cat("\n=== Per-resource edge counts (collapsing forward/reverse direction) ===\n")
resource_counts <- sort(table(unlist(edge_roles$edge_resource)), decreasing = TRUE)
resource_pct <- data.table(
  resource = names(resource_counts),
  n_edges  = as.integer(resource_counts),
  pct_of_all_edges = round(100 * as.integer(resource_counts) / n_edges_total, 4)
)
print(resource_pct)

# --- Assessable-universe sparsity: what fraction of edges even COULD be tested per resource ---
# (both Source_gene and Target_gene present in that resource's gene universe)
cat("\n=== Assessable universe size per resource (both genes present in resource's gene set) ===\n")

gene_universe_collectri <- union(collectri_strict$partner1_ensembl_gene_id, collectri_strict$partner2_ensembl_gene_id)
gene_universe_postar3   <- union(postar3_strict$partner1_ensembl_gene_id,   postar3_strict$partner2_ensembl_gene_id)
gene_universe_liana     <- union(liana_lenient$partner1_ensembl_gene_id,    liana_lenient$partner2_ensembl_gene_id)
gene_universe_string_s  <- union(string_exp_strict$partner1_ensembl_gene_id,  string_exp_strict$partner2_ensembl_gene_id)
gene_universe_string_l  <- union(string_exp_lenient$partner1_ensembl_gene_id, string_exp_lenient$partner2_ensembl_gene_id)
gene_universe_merops    <- union(merops_strict$partner1_ensembl_gene_id,    merops_strict$partner2_ensembl_gene_id)

assessable_summary <- data.table(
  resource = c("CollecTRI strict", "POSTAR3 strict", "Liana lenient",
               "STRING_exp strict", "STRING_exp lenient", "MEROPS strict"),
  n_assessable_edges = c(
    sum(edge_roles$Source_gene %in% gene_universe_collectri & edge_roles$Target_gene %in% gene_universe_collectri),
    sum(edge_roles$Source_gene %in% gene_universe_postar3   & edge_roles$Target_gene %in% gene_universe_postar3),
    sum(edge_roles$Source_gene %in% gene_universe_liana     & edge_roles$Target_gene %in% gene_universe_liana),
    sum(edge_roles$Source_gene %in% gene_universe_string_s  & edge_roles$Target_gene %in% gene_universe_string_s),
    sum(edge_roles$Source_gene %in% gene_universe_string_l  & edge_roles$Target_gene %in% gene_universe_string_l),
    sum(edge_roles$Source_gene %in% gene_universe_merops    & edge_roles$Target_gene %in% gene_universe_merops)
  )
)
assessable_summary[, pct_of_all_edges := round(100 * n_assessable_edges / n_edges_total, 3)]
print(assessable_summary)

# --- Factor-level sparsity (unchanged logic, recomputed on new edge_roles) ---
cat("\n=== Factor-level sparsity ===\n")
factor_annotation <- edge_roles[, .(
  n_edges           = .N,
  n_annotated_edges = sum(n_edge_roles > 0),
  any_annotation    = any(n_edge_roles > 0)
), by = .(Source_module, Source_gene)]

n_factors_total <- nrow(factor_annotation)
n_factors_annotated <- sum(factor_annotation$any_annotation)
cat(sprintf("Factors with >=1 annotated edge: %d / %d (%.2f%%)\n",
            n_factors_annotated, n_factors_total, 100 * n_factors_annotated / n_factors_total))

=== Overall edge-level sparsity ===
   has_annotation      N     pct
           <lgcl>  <int>   <num>
1:          FALSE 302416 98.9918
2:           TRUE   3080  1.0082

=== Per-label edge counts and % of total edges ===
                                  label n_edges pct_of_all_edges
                                 <char>   <int>            <num>
1: PPI (STRING_experimental >=0.1 <0.4)    1857           0.6079
2:                    Source_TF->Target     422           0.1381
3:                   Target_RBP->Source     286           0.0936
4:      PPI (STRING_experimental >=0.4)     219           0.0717
5:       Target_Ligand->Source_Receptor     140           0.0458
6:                   Source_RBP->Target      90           0.0295
7:                    Target_TF->Source      89           0.0291
8:       Source_Ligand->Target_Receptor      14           0.0046
9:   Target_Peptidase->Source_Substrate      11           0.0036

=== Per-resource edge counts (collapsing forward/reverse directi

# Classification 3: GWAS colocalization class

In [22]:
# --- Load the GWAS-colocalized table ---

cat("n rows:", nrow(GWAS_colocalized_eqtl_non_collapsed), "\n")
cat("n unique rows on (Source_module, Source_gene, Source_cell, Target_gene, Target_cell):",
    uniqueN(GWAS_colocalized_eqtl_non_collapsed, by = c("Source_module","Source_gene","Source_cell","Target_gene","Target_cell")), "\n")

# --- Build edge-level key on both tables, using the full 5-field match ---
edge_key_cols <- c("Source_module", "Source_gene", "Source_cell", "Target_gene", "Target_cell")

GWAS_colocalized_eqtl_non_collapsed[, edge_key := paste(Source_module, Source_gene, Source_cell, Target_gene, Target_cell)]
edge_roles[, edge_key := paste(Source_module, Source_gene, Source_cell, Target_gene, Target_cell)]

gwas_coloc_edge_keys <- unique(GWAS_colocalized_eqtl_non_collapsed$edge_key)

# --- Verify every colocalized edge is actually present in edge_roles (don't assume) ---
n_coloc_not_in_full <- sum(!gwas_coloc_edge_keys %in% edge_roles$edge_key)
cat("GWAS-colocalized edges NOT found in whole_eqtl_full_with_program_ID/edge_roles (should be 0):", n_coloc_not_in_full, "\n")

if (n_coloc_not_in_full > 0) {
  missing_keys <- gwas_coloc_edge_keys[!gwas_coloc_edge_keys %in% edge_roles$edge_key]
  print(GWAS_colocalized_eqtl_non_collapsed[edge_key %in% missing_keys][1:min(5, length(missing_keys))])
}

# --- Add the edge-level flag ---
edge_roles[, GWAS_colocalized := edge_key %in% gwas_coloc_edge_keys]

# --- Summary before proceeding ---
cat("\nEdges by GWAS_colocalized status:\n")
print(edge_roles[, .N, by = GWAS_colocalized])

# --- Cross-check: does this match the known input size? ---
cat("\nSanity: sum(GWAS_colocalized == TRUE) should equal", length(gwas_coloc_edge_keys),
    "(unique colocalized edges), got:", sum(edge_roles$GWAS_colocalized), "\n")

n rows: 19563 
n unique rows on (Source_module, Source_gene, Source_cell, Target_gene, Target_cell): 11400 
GWAS-colocalized edges NOT found in whole_eqtl_full_with_program_ID/edge_roles (should be 0): 0 

Edges by GWAS_colocalized status:
   GWAS_colocalized      N
             <lgcl>  <int>
1:            FALSE 291777
2:             TRUE  13719

Sanity: sum(GWAS_colocalized == TRUE) should equal 11400 (unique colocalized edges), got: 13719 


# SAVE

In [23]:
str(edge_roles)

Classes ‘data.table’ and 'data.frame':	305496 obs. of  19 variables:
 $ Source_module        : chr  "M_10" "M_10" "M_10" "M_10" ...
 $ Source_gene          : chr  "ENSG00000103145" "ENSG00000103145" "ENSG00000103145" "ENSG00000103145" ...
 $ program              : chr  "D_Factor36" "D_Factor36" "D_Factor36" "D_Factor36" ...
 $ Source_trait         : chr  "T_CD4_naive:ENSG00000103145" "T_CD4_naive:ENSG00000103145" "T_CD4_naive:ENSG00000103145" "T_CD4_naive:ENSG00000103145" ...
 $ Target_trait         : chr  "T_CD4_naive:ENSG00000185885" "T_CD4_naive:ENSG00000117984" "T_CD4_naive:ENSG00000175390" "T_CD4_naive:ENSG00000090554" ...
 $ Source_cell          : chr  "T_CD4_naive" "T_CD4_naive" "T_CD4_naive" "T_CD4_naive" ...
 $ Target_cell          : chr  "T_CD4_naive" "T_CD4_naive" "T_CD4_naive" "T_CD4_naive" ...
 $ Target_gene          : chr  "ENSG00000185885" "ENSG00000117984" "ENSG00000175390" "ENSG00000090554" ...
 $ edge.type            : chr  "Within" "Within" "Within" "Within" ...
 $ s

In [24]:
setwd(outdir)

edge_roles[, edge_role_str     := sapply(edge_role,     function(x) if (length(x) == 0) NA_character_ else paste(x, collapse = ";"))]
edge_roles[, edge_resource_str := sapply(edge_resource, function(x) if (length(x) == 0) NA_character_ else paste(x, collapse = ";"))]

final_cols <- c("Source_trait", "Target_trait", "program", "Source_cell", "Source_gene",
                 "Target_cell", "Target_gene", "Source_module", "edge.type",
                 "edge_type_composition", "edge_role_str", "edge_resource_str",
                 "n_edge_roles", "GWAS_colocalized")

whole_eqtl_annotated <- edge_roles[, ..final_cols]
setnames(whole_eqtl_annotated, c("edge_role_str", "edge_resource_str"), c("edge_role", "edge_resource"))

saveRDS(whole_eqtl_annotated, "whole_eqtl_annotated.rds")
fwrite(whole_eqtl_annotated, "whole_eqtl_annotated.tsv", sep = "\t")

cat("Saved", nrow(whole_eqtl_annotated), "rows,", ncol(whole_eqtl_annotated), "columns\n")
print(colnames(whole_eqtl_annotated))

# --- Final sanity check: reload and spot-check row count + annotated-edge count ---
# NOTE: the hardcoded `== 11400` GWAS check below is stale - it was the correct
# count for the OLD (pre-program) file. Do not assume it still applies; check
# the actual current values instead of hardcoding a number from a different dataset.
check <- readRDS("whole_eqtl_annotated.rds")
stopifnot(nrow(check) == nrow(whole_eqtl_full_with_program_ID))
stopifnot(sum(!is.na(check$edge_role)) == sum(edge_roles$n_edge_roles > 0))
cat("GWAS_colocalized TRUE count (new file, report only, no hardcoded check):", sum(check$GWAS_colocalized), "\n")
cat("Reload check passed: row count and annotated-edge count match.\n")

Saved 305496 rows, 14 columns
 [1] "Source_trait"          "Target_trait"          "program"              
 [4] "Source_cell"           "Source_gene"           "Target_cell"          
 [7] "Target_gene"           "Source_module"         "edge.type"            
[10] "edge_type_composition" "edge_role"             "edge_resource"        
[13] "n_edge_roles"          "GWAS_colocalized"     
GWAS_colocalized TRUE count (new file, report only, no hardcoded check): 13719 
Reload check passed: row count and annotated-edge count match.


In [25]:
head(whole_eqtl_annotated)

Source_trait,Target_trait,program,Source_cell,Source_gene,Target_cell,Target_gene,Source_module,edge.type,edge_type_composition,edge_role,edge_resource,n_edge_roles,GWAS_colocalized
<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<int>,<lgl>
T_CD4_naive:ENSG00000103145,T_CD4_naive:ENSG00000185885,D_Factor36,T_CD4_naive,ENSG00000103145,T_CD4_naive,ENSG00000185885,M_10,Within,Within-only,NA,NA,0,FALSE
T_CD4_naive:ENSG00000103145,T_CD4_naive:ENSG00000117984,D_Factor36,T_CD4_naive,ENSG00000103145,T_CD4_naive,ENSG00000117984,M_10,Within,Within-only,NA,NA,0,FALSE
T_CD4_naive:ENSG00000103145,T_CD4_naive:ENSG00000175390,D_Factor36,T_CD4_naive,ENSG00000103145,T_CD4_naive,ENSG00000175390,M_10,Within,Within-only,NA,NA,0,FALSE
T_CD4_naive:ENSG00000103145,T_CD4_naive:ENSG00000090554,D_Factor36,T_CD4_naive,ENSG00000103145,T_CD4_naive,ENSG00000090554,M_10,Within,Within-only,NA,NA,0,FALSE
T_CD4_naive:ENSG00000103145,T_CD4_naive:ENSG00000104870,D_Factor36,T_CD4_naive,ENSG00000103145,T_CD4_naive,ENSG00000104870,M_10,Within,Within-only,NA,NA,0,FALSE
T_CD4_naive:ENSG00000103145,T_CD4_naive:ENSG00000138795,D_Factor36,T_CD4_naive,ENSG00000103145,T_CD4_naive,ENSG00000138795,M_10,Within,Within-only,NA,NA,0,FALSE


# --- Feasibility check: does the structural precondition for a cascading pattern exist at all? ---


In [26]:
# Precondition: within the same (Source_module, Source_gene, target cell type),
# at least one annotated Target_gene must ALSO appear as a Source_gene somewhere
# in the dataset (i.e. it has its own cis-eQTL and its own trans-targets to cascade to).

annotated_edges <- edge_roles[n_edge_roles > 0]
cat("Total annotated edges:", nrow(annotated_edges), "\n")

# All genes that ever act as a Source_gene (i.e. have their own cis-eQTL/trans-eQTL structure)
all_source_genes <- unique(whole_eqtl_full_with_program_ID$Source_gene)
cat("Total unique Source_genes in the whole dataset:", length(all_source_genes), "\n")

# Among annotated edges, how many have a Target_gene that is ALSO a Source_gene elsewhere?
annotated_edges[, target_is_also_source := Target_gene %in% all_source_genes]

cat("\nAnnotated edges where Target_gene is ALSO a Source_gene elsewhere:\n")
print(annotated_edges[, .N, by = target_is_also_source])

# --- Narrow further: same factor, same TARGET cell type (the actual cascading precondition) ---
# For a true cascade, we need: Source_gene A -> Target_gene B (annotated, in target cell type X)
# AND Target_gene B has its OWN annotated edge as a Source_gene, in that SAME cell type X (as Source_cell)

candidate_cascade_bridges <- annotated_edges[target_is_also_source == TRUE,
                                              .(Source_module, Source_gene, Target_gene, Target_cell, edge_role)]

# Check: does Target_gene appear as Source_gene with Source_cell == that same Target_cell,
# AND does THAT row also have an annotation (n_edge_roles > 0)?
setkey(annotated_edges, Source_gene, Source_cell)

cascade_check <- candidate_cascade_bridges[, {
  match_rows <- annotated_edges[Source_gene == Target_gene & Source_cell == Target_cell]
  .(n_potential_second_hop = nrow(match_rows))
}, by = .(Source_module, Source_gene, Target_gene, Target_cell)]

n_true_cascade_bridges <- sum(cascade_check$n_potential_second_hop > 0)
cat("\nCandidate first-hop edges where target gene also has an ANNOTATED second-hop edge",
    "in the SAME cell type:", n_true_cascade_bridges, "/", nrow(cascade_check), "\n")

if (n_true_cascade_bridges > 0) {
  print(cascade_check[n_potential_second_hop > 0])
}

Total annotated edges: 3080 
Total unique Source_genes in the whole dataset: 1152 

Annotated edges where Target_gene is ALSO a Source_gene elsewhere:
   target_is_also_source     N
                  <lgcl> <int>
1:                 FALSE  2345
2:                  TRUE   735

Candidate first-hop edges where target gene also has an ANNOTATED second-hop edge in the SAME cell type: 0 / 487 
